# Bu Dersi Google Colab'da Çalıştır

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BILSEM-BT/Python/blob/main/26-PythonYapayZekaTahminProjesi.ipynb)

Bu notebook GitHub üzerinde ders dokümanı olarak yayımlanır. Kodları çalıştırmak ve üzerinde denemeler yapmak için yukarıdaki **Open in Colab** butonunu kullanabilirsiniz.

### Nasıl çalışacağız?

1. **Open in Colab** butonuna tıklayın.
2. Açılan notebook'taki kod hücrelerini `▶` düğmesiyle çalıştırın.
3. Kodları değiştirerek farklı sonuçları deneyin.
4. Çalışmalarınız kendi Colab çalışma alanınızda tutulur; bu GitHub'daki ana ders dosyasını değiştirmez.

> **Önemli:** GitHub'daki bu dosya dersin ana ve değiştirilmeyen kaynağıdır. Colab'da yaptığınız değişiklikler bu dosyaya otomatik olarak yazılmaz.

---

# 26 - Python ile Uçtan Uca Yapay Zeka Tahmin Projesi

## El Yazısı Rakam Tanıma Sistemi

**Niyazi Sayın BİLSEM**  
**Bilişim Teknolojileri Dersi**  
**Ders Öğretmeni: Ersin ŞANLI**

Bu derste şimdiye kadar öğrendiğimiz yapay zeka konularını tek bir proje içinde birleştireceğiz.

Projemizin amacı:

**8×8 piksel boyutundaki el yazısı rakam görüntülerinden 0-9 arasındaki rakamı tahmin eden bir yapay zeka modeli geliştirmek.**

Bu proje boyunca:

- gerçek bir veri kümesini yükleyeceğiz,
- görüntü verisinin sayısal temsilini inceleyeceğiz,
- veri analizi yapacağız,
- final test setini baştan ayıracağız,
- baseline model oluşturacağız,
- birden fazla sınıflandırma algoritması deneyeceğiz,
- Cross Validation ile modelleri karşılaştıracağız,
- hiperparametre optimizasyonu yapacağız,
- final test performansını ölçeceğiz,
- confusion matrix oluşturacağız,
- yanlış tahminleri inceleyeceğiz,
- yeni bir görüntü için tahmin üreteceğiz,
- eğitilmiş modeli dosyaya kaydedeceğiz.

Bu ders, önceki yapay zeka derslerinin uygulama projesidir.


# 1. Proje Problemi

Bilgisayara aşağıdaki gibi küçük bir el yazısı rakam görüntüsü verildiğini düşünelim:

```text
Görüntü
↓
Piksel Değerleri
↓
Makine Öğrenmesi Modeli
↓
Tahmin
↓
0, 1, 2, ..., 9
```

Bu bir **çok sınıflı sınıflandırma** problemidir.

Hedef sınıflar:

```text
0
1
2
3
4
5
6
7
8
9
```

olacaktır.


# 2. Neden Bu Projeyi Seçtik?

Bu proje birçok yapay zeka kavramını aynı anda kullanmamızı sağlar:

- gerçek veri,
- görüntü temsili,
- çok sınıflı sınıflandırma,
- model karşılaştırma,
- Cross Validation,
- hiperparametre optimizasyonu,
- hata analizi,
- model kaydetme,
- ileride masaüstü veya web arayüzüne bağlama.

Ayrıca veri kümesi scikit-learn içinde hazır bulunduğu için internet bağlantısına ihtiyaç duymaz.


# 3. Digits Veri Kümesi

Scikit-learn'in Digits veri kümesi, 0-9 arasındaki el yazısı rakamların küçük gri tonlu görüntülerinden oluşur.

Her görüntü:

```text
8 × 8 = 64
```

piksel içerir.

Makine öğrenmesi modeli bu 64 piksel değerini özellik olarak kullanabilir.


# 4. Gerekli Kütüphaneler

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits


# 5. Veri Kümesini Yüklemek

In [ ]:
digits = load_digits()

print(
    digits.keys()
)


Veri kümesinde önemli bölümler:

- `data` → 64 özellikli sayısal tablo,
- `target` → gerçek rakam,
- `images` → 8×8 görüntü biçimi,
- `target_names` → sınıf isimleri.


# 6. Veri Kümesinin Boyutu

In [ ]:
print(
    "Data boyutu:",
    digits.data.shape
)

print(
    "Target boyutu:",
    digits.target.shape
)

print(
    "Images boyutu:",
    digits.images.shape
)


Burada:

```text
1797 örnek
64 özellik
```

bulunmaktadır.

Her örnek ayrıca 8×8 görüntü olarak saklanmaktadır.


# 7. Özellik ve Hedef Değişkenleri

In [ ]:
X = pd.DataFrame(
    digits.data,
    columns=[
        f"Piksel{i}"
        for i in range(64)
    ]
)

y = pd.Series(
    digits.target,
    name="Rakam"
)

X.head()


# 8. Hedef Değerler

In [ ]:
print(
    sorted(
        y.unique()
    )
)


Modelimizin tahmin edeceği 10 farklı sınıf vardır.


# 9. İlk Görüntüyü İncelemek

In [ ]:
plt.imshow(
    digits.images[0]
)

plt.title(
    f"Gerçek Rakam: {digits.target[0]}"
)

plt.axis("off")
plt.show()


Görüntü bilgisayarda aslında bir sayı matrisi olarak tutulmaktadır.


# 10. İlk Görüntünün Piksel Matrisi

In [ ]:
print(
    digits.images[0]
)


Her hücre bir piksel yoğunluğunu temsil eder.

Bu veri kümesinde piksel değerleri 0 ile 16 arasında değişir.


# 11. İlk Görüntünün 64 Özellikli Hali

In [ ]:
print(
    digits.data[0]
)


8×8 matris model için 64 elemanlı tek boyutlu özellik dizisine dönüştürülmüştür.


# 12. Birkaç Farklı Rakamı Görmek

Her görüntüyü ayrı ayrı inceleyelim.


In [ ]:
ornek_indexleri = [
    0,
    10,
    20,
    30,
    40
]

for index in ornek_indexleri:
    plt.figure()

    plt.imshow(
        digits.images[index]
    )

    plt.title(
        f"Gerçek Rakam: {digits.target[index]}"
    )

    plt.axis("off")
    plt.show()


# 13. Sınıf Dağılımı

In [ ]:
sinif_sayilari = (
    y.value_counts()
    .sort_index()
)

sinif_sayilari


# 14. Sınıf Dağılım Grafiği

In [ ]:
plt.bar(
    sinif_sayilari.index.astype(str),
    sinif_sayilari.values
)

plt.xlabel("Rakam")
plt.ylabel("Örnek Sayısı")
plt.title("Digits Sınıf Dağılımı")
plt.show()


Sınıfların örnek sayıları birbirine oldukça yakındır.


# 15. Eksik Veri Kontrolü

In [ ]:
print(
    X.isna().sum().sum()
)


# 16. Piksel Değerlerinin Aralığı

In [ ]:
print(
    "En küçük piksel:",
    X.min().min()
)

print(
    "En büyük piksel:",
    X.max().max()
)


# 17. Piksel Ortalama Değerleri

In [ ]:
print(
    X.mean().head(10)
)


Bazı piksel konumları görüntülerin çoğunda sürekli sıfıra yakın olabilir.

Bu durum rakamların görüntü içinde belirli bölgelerde yoğunlaşmasından kaynaklanabilir.


# 18. Tamamen Sabit Özellikler Var mı?

Hiç değişmeyen piksel sütunlarını bulalım.


In [ ]:
sabit_sutunlar = [
    sutun
    for sutun in X.columns
    if X[sutun].nunique() == 1
]

print(
    "Sabit sütun sayısı:",
    len(sabit_sutunlar)
)

print(
    sabit_sutunlar
)


Sabit sütunlar modele bilgi sağlamaz.

Bu projede modellerin aynı giriş yapısını koruması için 64 özellikli orijinal veriyle devam edeceğiz.


# 19. Final Test Setini Başta Ayırmak

Model seçimi sırasında final test setine bakmayacağız.

Verinin %20'sini final test için ayıralım.


In [ ]:
from sklearn.model_selection import train_test_split

X_gelistirme, X_test, y_gelistirme, y_test, img_gelistirme, img_test = train_test_split(
    X,
    y,
    digits.images,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(
    "Geliştirme:",
    X_gelistirme.shape
)

print(
    "Final test:",
    X_test.shape
)


Bu derste görüntü dizilerini de aynı bölmeyle ayırdık.

Böylece final testte yanlış tahmin edilen örneklerin gerçek görüntülerini daha sonra gösterebiliriz.


# 20. Geliştirme ve Test Sınıf Oranları

In [ ]:
print(
    "Geliştirme:"
)

print(
    y_gelistirme.value_counts(
        normalize=True
    ).sort_index()
)

print()

print(
    "Final test:"
)

print(
    y_test.value_counts(
        normalize=True
    ).sort_index()
)


# 21. Baseline Model

Gerçek modelleri değerlendirmeden önce basit bir başlangıç modeli oluşturalım.


In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(
    strategy="most_frequent"
)

baseline.fit(
    X_gelistirme,
    y_gelistirme
)

baseline_accuracy = baseline.score(
    X_test,
    y_test
)

print(
    "Baseline Accuracy:",
    baseline_accuracy
)


10 sınıflı bir problemde sürekli tek bir sınıfı tahmin eden modelin başarısı düşüktür.

Gerçek modellerimizin bu baseline'dan belirgin biçimde daha iyi olması gerekir.


# 22. Cross Validation Planı

Çok sınıflı sınıflandırmada sınıf oranlarını korumak için StratifiedKFold kullanacağız.


In [ ]:
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print(cv)


# 23. Kullanacağımız Değerlendirme Metrikleri

Çok sınıflı sınıflandırmada:

- accuracy,
- macro precision,
- macro recall,
- macro F1

kullanacağız.

`macro` yaklaşımı her sınıf için metriği hesaplayıp sınıflara eşit ağırlık verir.


# 24. Model 1: Logistic Regression

Piksel özelliklerini StandardScaler ile ölçeklendirerek Logistic Regression modeli oluşturalım.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

lojistik = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=3000,
            random_state=42
        )
    )
])

print(lojistik)


# 25. Logistic Regression Cross Validation

In [ ]:
skorlar = {
    "accuracy": "accuracy",
    "precision": "precision_macro",
    "recall": "recall_macro",
    "f1": "f1_macro"
}

lojistik_cv = cross_validate(
    lojistik,
    X_gelistirme,
    y_gelistirme,
    cv=cv,
    scoring=skorlar,
    n_jobs=-1
)

print(
    "Ortalama Accuracy:",
    lojistik_cv[
        "test_accuracy"
    ].mean()
)

print(
    "Ortalama Macro F1:",
    lojistik_cv[
        "test_f1"
    ].mean()
)


# 26. Model 2: KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        KNeighborsClassifier(
            n_neighbors=5
        )
    )
])

knn_cv = cross_validate(
    knn,
    X_gelistirme,
    y_gelistirme,
    cv=cv,
    scoring=skorlar,
    n_jobs=-1
)

print(
    "KNN Accuracy:",
    knn_cv[
        "test_accuracy"
    ].mean()
)

print(
    "KNN Macro F1:",
    knn_cv[
        "test_f1"
    ].mean()
)


# 27. Model 3: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

orman = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

orman_cv = cross_validate(
    orman,
    X_gelistirme,
    y_gelistirme,
    cv=cv,
    scoring=skorlar,
    n_jobs=-1
)

print(
    "Random Forest Accuracy:",
    orman_cv[
        "test_accuracy"
    ].mean()
)

print(
    "Random Forest Macro F1:",
    orman_cv[
        "test_f1"
    ].mean()
)


# 28. Model 4: Support Vector Machine

In [ ]:
from sklearn.svm import SVC

svm = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        SVC(
            kernel="rbf",
            C=3,
            gamma="scale"
        )
    )
])

svm_cv = cross_validate(
    svm,
    X_gelistirme,
    y_gelistirme,
    cv=cv,
    scoring=skorlar,
    n_jobs=-1
)

print(
    "SVM Accuracy:",
    svm_cv[
        "test_accuracy"
    ].mean()
)

print(
    "SVM Macro F1:",
    svm_cv[
        "test_f1"
    ].mean()
)


# 29. Model Sonuçlarını Fonksiyonla Özetlemek

In [ ]:
def cv_ozet(
    model_adi,
    cv_sonucu
):
    return {
        "Model": model_adi,
        "Accuracy": cv_sonucu[
            "test_accuracy"
        ].mean(),
        "PrecisionMacro": cv_sonucu[
            "test_precision"
        ].mean(),
        "RecallMacro": cv_sonucu[
            "test_recall"
        ].mean(),
        "F1Macro": cv_sonucu[
            "test_f1"
        ].mean(),
        "F1Std": cv_sonucu[
            "test_f1"
        ].std()
    }


# 30. Model Karşılaştırma Tablosu

In [ ]:
model_karsilastirma = pd.DataFrame([
    cv_ozet(
        "Logistic Regression",
        lojistik_cv
    ),
    cv_ozet(
        "KNN",
        knn_cv
    ),
    cv_ozet(
        "Random Forest",
        orman_cv
    ),
    cv_ozet(
        "SVM",
        svm_cv
    )
])

model_karsilastirma.sort_values(
    "F1Macro",
    ascending=False
)


# 31. Modellerin Accuracy Grafiği

In [ ]:
sirali = model_karsilastirma.sort_values(
    "Accuracy",
    ascending=False
)

plt.bar(
    sirali["Model"],
    sirali["Accuracy"]
)

plt.ylim(0, 1)
plt.ylabel("Cross Validation Accuracy")
plt.title("Model Karşılaştırması")
plt.xticks(rotation=30)
plt.show()


# 32. Modellerin Macro F1 Grafiği

In [ ]:
sirali_f1 = model_karsilastirma.sort_values(
    "F1Macro",
    ascending=False
)

plt.bar(
    sirali_f1["Model"],
    sirali_f1["F1Macro"]
)

plt.ylim(0, 1)
plt.ylabel("Cross Validation Macro F1")
plt.title("Model Karşılaştırması")
plt.xticks(rotation=30)
plt.show()


# 33. Neden Final Test Setine Hâlâ Bakmıyoruz?

Model karşılaştırma ve hiperparametre seçimi devam ediyor.

Final test setine şimdi bakarsak seçim sürecimizi test sonuçlarına göre etkilemiş oluruz.

Bu nedenle önce geliştirme verisi içinde en iyi modeli belirleyeceğiz.


# 34. SVM Hiperparametre Optimizasyonu

Cross Validation sonuçlarında güçlü bir aday olan SVM için GridSearchCV uygulayalım.

Arayacağımız temel hiperparametreler:

```text
C
gamma
```


In [ ]:
from sklearn.model_selection import GridSearchCV

svm_grid = {
    "model__C": [
        1,
        3,
        10
    ],
    "model__gamma": [
        "scale",
        0.001,
        0.01
    ]
}

svm_arama = GridSearchCV(
    estimator=svm,
    param_grid=svm_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1
)

svm_arama.fit(
    X_gelistirme,
    y_gelistirme
)

print(
    svm_arama.best_params_
)

print(
    "En iyi CV Macro F1:",
    svm_arama.best_score_
)


# 35. Grid Search Sonuçları

In [ ]:
svm_grid_sonuclari = pd.DataFrame(
    svm_arama.cv_results_
)

svm_grid_ozet = svm_grid_sonuclari[
    [
        "param_model__C",
        "param_model__gamma",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values(
    "rank_test_score"
)

svm_grid_ozet


# 36. En İyi SVM Modeli

In [ ]:
en_iyi_model = (
    svm_arama.best_estimator_
)

print(
    en_iyi_model
)


# 37. Final Test Aşaması

Model seçimi tamamlandı.

Artık başta ayırdığımız final test setini ilk kez kullanabiliriz.


In [ ]:
final_tahmin = (
    en_iyi_model.predict(
        X_test
    )
)

print(
    final_tahmin[:20]
)


# 38. Final Test Accuracy

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

final_accuracy = accuracy_score(
    y_test,
    final_tahmin
)

print(
    "Final Accuracy:",
    final_accuracy
)


# 39. Final Macro Precision

In [ ]:
final_precision = precision_score(
    y_test,
    final_tahmin,
    average="macro"
)

print(
    "Macro Precision:",
    final_precision
)


# 40. Final Macro Recall

In [ ]:
final_recall = recall_score(
    y_test,
    final_tahmin,
    average="macro"
)

print(
    "Macro Recall:",
    final_recall
)


# 41. Final Macro F1

In [ ]:
final_f1 = f1_score(
    y_test,
    final_tahmin,
    average="macro"
)

print(
    "Macro F1:",
    final_f1
)


# 42. Final Sonuç Özeti

In [ ]:
final_ozet = pd.DataFrame({
    "Metrik": [
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1"
    ],
    "Deger": [
        final_accuracy,
        final_precision,
        final_recall,
        final_f1
    ]
})

final_ozet


# 43. Classification Report

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        final_tahmin,
        digits=3
    )
)


Classification report her rakam sınıfı için:

- precision,
- recall,
- F1-score,
- support

bilgilerini gösterir.


# 44. Confusion Matrix

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay
)

cm = confusion_matrix(
    y_test,
    final_tahmin
)

print(cm)


# 45. Confusion Matrix Görseli

In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=digits.target_names
).plot()

plt.title(
    "El Yazısı Rakam Tanıma Confusion Matrix"
)

plt.show()


Ana köşegen doğru tahminleri gösterir.

Köşegen dışındaki değerler modelin hangi rakamları birbirine karıştırdığını anlamamıza yardımcı olur.


# 46. En Çok Karıştırılan Sınıfları Bulmak

Ana köşegeni sıfırlayıp en büyük yanlış sınıflandırma sayılarını inceleyelim.


In [ ]:
cm_hata = cm.copy()

np.fill_diagonal(
    cm_hata,
    0
)

hata_listesi = []

for gercek in range(10):
    for tahmin in range(10):
        adet = cm_hata[
            gercek,
            tahmin
        ]

        if adet > 0:
            hata_listesi.append({
                "Gercek": gercek,
                "Tahmin": tahmin,
                "Adet": adet
            })

hata_df = pd.DataFrame(
    hata_listesi
).sort_values(
    "Adet",
    ascending=False
)

hata_df.head(10)


# 47. Yanlış Tahminleri Bulmak

In [ ]:
yanlis_mask = (
    y_test.to_numpy()
    != final_tahmin
)

yanlis_indexler = np.where(
    yanlis_mask
)[0]

print(
    "Yanlış tahmin sayısı:",
    len(yanlis_indexler)
)


# 48. İlk Yanlış Tahmini İncelemek

In [ ]:
if len(yanlis_indexler) > 0:
    i = yanlis_indexler[0]

    plt.imshow(
        img_test[i]
    )

    plt.title(
        f"Gerçek: {y_test.iloc[i]} - Tahmin: {final_tahmin[i]}"
    )

    plt.axis("off")
    plt.show()
else:
    print(
        "Yanlış tahmin bulunamadı."
    )


# 49. Birkaç Yanlış Tahmini Ayrı Ayrı Görmek

In [ ]:
for i in yanlis_indexler[:5]:
    plt.figure()

    plt.imshow(
        img_test[i]
    )

    plt.title(
        f"Gerçek: {y_test.iloc[i]} - Tahmin: {final_tahmin[i]}"
    )

    plt.axis("off")
    plt.show()


Yanlış tahminleri incelemek model geliştirme sürecinin önemli bir parçasıdır.

Sadece toplam başarı skoruna bakmak yerine modelin nerede zorlandığını anlamaya çalışırız.


# 50. Doğru Tahminlerden Bir Örnek

In [ ]:
dogru_indexler = np.where(
    y_test.to_numpy()
    == final_tahmin
)[0]

i = dogru_indexler[0]

plt.imshow(
    img_test[i]
)

plt.title(
    f"Gerçek: {y_test.iloc[i]} - Tahmin: {final_tahmin[i]}"
)

plt.axis("off")
plt.show()


# 51. Tek Bir Test Görüntüsü İçin Tahmin

Bir final test örneğini modele tek başına verelim.


In [ ]:
ornek = X_test.iloc[[0]]

ornek_tahmin = en_iyi_model.predict(
    ornek
)[0]

print(
    "Gerçek:",
    y_test.iloc[0]
)

print(
    "Tahmin:",
    ornek_tahmin
)


# 52. Tahmin Edilen Görüntüyü Gösterelim

In [ ]:
plt.imshow(
    img_test[0]
)

plt.title(
    f"Tahmin: {ornek_tahmin}"
)

plt.axis("off")
plt.show()


# 53. SVM Karar Skorları

SVC modelimiz `decision_function()` ile her sınıf için karar skorları üretebilir.


In [ ]:
karar_skorlari = (
    en_iyi_model.decision_function(
        ornek
    )[0]
)

skor_df = pd.DataFrame({
    "Rakam": range(10),
    "KararSkoru": karar_skorlari
}).sort_values(
    "KararSkoru",
    ascending=False
)

skor_df


En yüksek karar skoru modelin seçtiği sınıfla ilişkilidir.

Karar skorlarını doğrudan olasılık olarak yorumlamamalıyız.


# 54. En Güçlü İlk 3 Sınıf Adayı

In [ ]:
skor_df.head(3)


# 55. Modelin Bütün Test Tahminlerini DataFrame'de İncelemek

In [ ]:
tahmin_df = pd.DataFrame({
    "Gercek": y_test.to_numpy(),
    "Tahmin": final_tahmin
})

tahmin_df["DogruMu"] = (
    tahmin_df["Gercek"]
    == tahmin_df["Tahmin"]
)

tahmin_df.head(20)


# 56. Rakam Bazında Doğru Tahmin Oranı

In [ ]:
rakam_basarisi = (
    tahmin_df
    .groupby("Gercek")[
        "DogruMu"
    ]
    .mean()
)

rakam_basarisi


# 57. Rakam Bazında Başarı Grafiği

In [ ]:
plt.bar(
    rakam_basarisi.index.astype(str),
    rakam_basarisi.values
)

plt.ylim(0, 1)
plt.xlabel("Gerçek Rakam")
plt.ylabel("Doğru Tahmin Oranı")
plt.title("Rakam Bazında Model Başarısı")
plt.show()


Bu grafik modelin bazı rakamlarda diğerlerinden daha fazla zorlanıp zorlanmadığını görmemizi sağlar.


# 58. Modeli Kaydetmek

Eğitilmiş modeli tekrar tekrar eğitmek yerine dosyaya kaydedelim.


In [ ]:
import joblib

MODEL_DOSYASI = (
    "26-rakam-tanima-modeli.joblib"
)

joblib.dump(
    en_iyi_model,
    MODEL_DOSYASI
)

print(
    "Model kaydedildi:",
    MODEL_DOSYASI
)


# 59. Modeli Yeniden Yüklemek

In [ ]:
yuklenen_model = joblib.load(
    MODEL_DOSYASI
)

print(
    yuklenen_model
)


# 60. Yüklenen Modelle Tahmin

In [ ]:
yeniden_tahmin = (
    yuklenen_model.predict(
        X_test.iloc[[1]]
    )[0]
)

print(
    "Gerçek:",
    y_test.iloc[1]
)

print(
    "Tahmin:",
    yeniden_tahmin
)


# 61. Tahmin Fonksiyonu Oluşturmak

64 piksel değerini alan bir tahmin fonksiyonu yazalım.


In [ ]:
def rakam_tahmin(
    model,
    piksel_degerleri
):
    if len(
        piksel_degerleri
    ) != 64:
        raise ValueError(
            "Tam olarak 64 piksel değeri gereklidir."
        )

    veri = pd.DataFrame(
        [piksel_degerleri],
        columns=X.columns
    )

    tahmin = model.predict(
        veri
    )[0]

    return int(tahmin)


# 62. Tahmin Fonksiyonunu Test Etmek

In [ ]:
test_piksel = (
    X_test.iloc[2]
    .to_numpy()
)

print(
    "Gerçek:",
    y_test.iloc[2]
)

print(
    "Tahmin:",
    rakam_tahmin(
        yuklenen_model,
        test_piksel
    )
)


# 63. Görüntüden 64 Özelliğe Dönüşüm

Elimizde 8×8 bir görüntü matrisi varsa:


In [ ]:
ornek_goruntu = (
    img_test[2]
)

print(
    ornek_goruntu.shape
)

duz_veri = (
    ornek_goruntu.reshape(-1)
)

print(
    duz_veri.shape
)


`reshape(-1)` ile 8×8 görüntüyü 64 özellikli diziye dönüştürdük.


# 64. Görüntüden Tahmin Fonksiyonu

In [ ]:
def goruntuden_rakam_tahmin(
    model,
    goruntu
):
    if goruntu.shape != (
        8,
        8
    ):
        raise ValueError(
            "Görüntü 8x8 boyutunda olmalıdır."
        )

    piksel = goruntu.reshape(
        1,
        -1
    )

    veri = pd.DataFrame(
        piksel,
        columns=X.columns
    )

    return int(
        model.predict(
            veri
        )[0]
    )


In [ ]:
print(
    "Gerçek:",
    y_test.iloc[2]
)

print(
    "Tahmin:",
    goruntuden_rakam_tahmin(
        yuklenen_model,
        img_test[2]
    )
)


# 65. Gerçek Bir Kullanıcı Çizimi Nasıl Kullanılır?

Digits veri kümesindeki görüntüler:

```text
8×8
0-16 piksel yoğunluğu
```

biçimindedir.

Bir kullanıcıdan alınan normal PNG veya JPG görüntüsü doğrudan modele verilmemelidir.

Önce eğitim verisiyle aynı biçime dönüştürülmesi gerekir:

```text
Görüntüyü al
↓
Gri tona dönüştür
↓
Rakamı uygun biçimde merkezle
↓
8×8 boyutuna getir
↓
Piksel ölçeğini eğitim verisine uyarla
↓
64 özelliğe dönüştür
↓
Model.predict()
```

Bu işlemleri ileride görüntü işleme derslerinde uygulayacağız.


# 66. Model Eğitim Verisi ile Uygulama Verisi Aynı Yapıda Olmalı

Makine öğrenmesi uygulamalarındaki en önemli kurallardan biri:

**Tahmin sırasında modele verilen veri, eğitimde kullanılan veri yapısına uygun olmalıdır.**

Örneğin model:

```text
8×8
0-16 piksel
64 özellik
```

ile eğitildiyse tahmin verisi de aynı yapıya hazırlanmalıdır.


# 67. Veri Dağılımı Değişirse Ne Olur?

Gerçek kullanıcıların çizdiği rakamlar Digits veri kümesindeki örneklerden çok farklıysa model performansı düşebilir.

Örneğin:

- farklı çözünürlük,
- farklı çizgi kalınlığı,
- farklı konum,
- farklı parlaklık,
- farklı görüntü arka planı

modeli etkileyebilir.

Bu durum bize eğitim verisinin gerçek kullanım ortamını temsil etmesinin önemini gösterir.


# 68. Model Başarısı Yüzde 100 Olmak Zorunda mı?

Gerçek yapay zeka uygulamalarında hata olması normaldir.

Önemli olan:

- modelin hangi durumlarda hata yaptığını bilmek,
- hata maliyetini değerlendirmek,
- veri kalitesini artırmak,
- uygun modeli seçmek,
- gerçek kullanım koşullarında test etmektir.


# 69. Modeli Daha İyi Hale Getirmek İçin Neler Yapabiliriz?

Olası geliştirmeler:

- daha fazla eğitim verisi,
- daha çeşitli el yazısı örnekleri,
- görüntü ön işleme,
- farklı özellik çıkarma yöntemleri,
- PCA,
- farklı modeller,
- veri artırma,
- yapay sinir ağları,
- convolutional neural network.

İlerleyen derslerde bu yöntemlerin bazılarını uygulayacağız.


# 70. PCA ile Boyut Azaltmaya İlk Bakış

64 piksel özelliğini daha az sayıda bileşenle temsil etmek mümkün olabilir.

PCA bunu yapabilen yöntemlerden biridir.

Bu projede PCA'yı model zorunluluğu olarak kullanmayacağız; yalnızca veri yapısını incelemek için küçük bir örnek yapacağız.


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(
    n_components=2
)

X_pca = pca.fit_transform(
    X_gelistirme
)

print(
    X_pca.shape
)


# 71. PCA ile İki Boyutlu Görünüm

10 sınıfı iki bileşen üzerinde görelim.


In [ ]:
pca_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "Rakam": y_gelistirme.to_numpy()
})

for rakam in range(10):
    secim = (
        pca_df["Rakam"]
        == rakam
    )

    plt.scatter(
        pca_df.loc[
            secim,
            "PC1"
        ],
        pca_df.loc[
            secim,
            "PC2"
        ],
        label=str(rakam)
    )

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Digits PCA Görünümü")
plt.legend()
plt.show()


İki boyuta indirildiğinde bazı sınıflar iç içe görünebilir.

Asıl modellerimiz 64 özelliğin tamamını kullanmaktadır.


# 72. PCA Açıklanan Varyans

In [ ]:
print(
    pca.explained_variance_ratio_
)

print(
    "Toplam açıklanan varyans:",
    pca.explained_variance_ratio_.sum()
)


İki bileşen bütün bilgiyi taşımaz.

Bu nedenle yalnızca iki boyutlu görsele bakarak veri kümesinin tam sınıflandırılabilirliğini değerlendirmemeliyiz.


# 73. Proje Sonuç Raporu Fonksiyonu

Modelimizin temel sonuçlarını bir sözlük olarak üretelim.


In [ ]:
def proje_raporu():
    return {
        "ornek_sayisi":
            int(
                len(X)
            ),
        "ozellik_sayisi":
            int(
                X.shape[1]
            ),
        "sinif_sayisi":
            int(
                y.nunique()
            ),
        "en_iyi_cv_f1":
            round(
                float(
                    svm_arama.best_score_
                ),
                4
            ),
        "final_accuracy":
            round(
                float(
                    final_accuracy
                ),
                4
            ),
        "final_macro_f1":
            round(
                float(
                    final_f1
                ),
                4
            ),
        "yanlis_tahmin":
            int(
                len(
                    yanlis_indexler
                )
            )
    }


In [ ]:
rapor = proje_raporu()

print(rapor)


# 74. Sonuçları JSON Dosyasına Kaydetmek

In [ ]:
import json

with open(
    "26-proje-raporu.json",
    "w",
    encoding="utf-8"
) as dosya:
    json.dump(
        rapor,
        dosya,
        ensure_ascii=False,
        indent=4
    )

print(
    "Proje raporu kaydedildi."
)


# 75. Tahmin Sonuçlarını CSV'ye Kaydetmek

In [ ]:
tahmin_df.to_csv(
    "26-final-test-tahminleri.csv",
    index=False,
    encoding="utf-8"
)

print(
    "Tahmin sonuçları kaydedildi."
)


# 76. Yapay Zeka Projesinde Dosya Yapısı

Gerçek bir proje şu şekilde düzenlenebilir:

```text
rakam_tanima/
│
├── model_egit.py
├── tahmin.py
├── model/
│   └── rakam_modeli.joblib
├── rapor/
│   ├── proje_raporu.json
│   └── tahminler.csv
└── app.py
```

Bu yapı eğitim kodu ile uygulama kodunu ayırmaya yardımcı olur.


# 77. Eğitim ve Tahmin Kodunu Ayırmak

Gerçek uygulamada:

### `model_egit.py`

- veriyi yükler,
- modeli eğitir,
- model seçer,
- final test yapar,
- `.joblib` kaydeder.

### `tahmin.py`

- kayıtlı modeli yükler,
- yeni veriyi hazırlar,
- tahmin üretir.

Bu ayrım uygulamanın her tahminde modeli yeniden eğitmesini önler.


# 78. Flask ile Rakam Tanıma Uygulaması

İleride web uygulaması akışımız:

```text
Kullanıcı Rakam Görüntüsü Yükler
↓
Flask
↓
Görüntü Ön İşleme
↓
8×8 / 64 Özellik
↓
Model
↓
Tahmin
↓
HTML Sonuç
```

şeklinde olabilir.


# 79. Tkinter ile Rakam Tanıma Uygulaması

Masaüstü sürümünde kullanıcı:

```text
Canvas üzerine rakam çizer
↓
Görüntü alınır
↓
8×8'e dönüştürülür
↓
Model.predict()
↓
Tahmin Label'da gösterilir
```

şeklinde bir uygulama kullanabilir.

Bu uygulamayı görüntü işleme derslerinden sonra geliştirmek daha uygun olacaktır.


# 80. Yapay Zeka Modelini API Olarak Sunmak

Bir API modeli:

```text
POST /predict
```

adresinde çalıştırabilir.

İstek:

```json
{
    "pixels": [0, 0, 5, 13, ...]
}
```

Cevap:

```json
{
    "prediction": 5
}
```

olabilir.

Daha sonraki derslerde Flask ile yapay zeka API uygulaması oluşturacağız.


# 81. Gerçek Projede İzlenmesi Gereken Süreç

```text
1. Problemi Tanımla
2. Veriyi Tanı
3. Veri Kalitesini Kontrol Et
4. Final Test Setini Ayır
5. Baseline Oluştur
6. Birden Fazla Model Dene
7. Cross Validation Yap
8. Uygun Metrikleri Seç
9. Hiperparametreleri Optimize Et
10. Final Test Yap
11. Hataları Analiz Et
12. Modeli Kaydet
13. Tahmin Fonksiyonu Yaz
14. Uygulamaya Entegre Et
15. Gerçek Kullanım Verisinde İzle
```

Bu akış profesyonel makine öğrenmesi çalışmalarının temel düşüncesini oluşturur.


# 82. Model Seçerken Sadece Accuracy Kullanmadık

Bu proje çok sınıflı bir sınıflandırma problemidir.

Bu nedenle:

- accuracy,
- macro precision,
- macro recall,
- macro F1,
- confusion matrix,
- sınıf bazlı rapor,
- yanlış tahmin analizi

birlikte kullanıldı.

Bir modelin performansını tek bir sayıya indirgememek önemlidir.


# 83. Veri Sızıntısından Nasıl Kaçındık?

Bu projede:

- final test setini en başta ayırdık,
- scaler'ı Pipeline içine koyduk,
- Grid Search yalnızca geliştirme verisinde yapıldı,
- final test yalnızca model seçimi bittikten sonra kullanıldı.

Bu adımlar değerlendirme güvenilirliğini artırır.


# 84. Model Dosyası Güvenliği

`joblib` ve pickle tabanlı dosyalar yalnızca güvenilen kaynaklardan yüklenmelidir.

İnternetten indirilen bilinmeyen model dosyaları uygulamada doğrudan açılmamalıdır.


# 85. Etik ve Sorumlu Yapay Zeka

Bu proje yalnızca el yazısı rakam tanımaktadır.

Ancak gerçek yapay zeka projelerinde şu sorular sorulmalıdır:

- Veri kişisel bilgi içeriyor mu?
- Veri kullanım izni var mı?
- Model bazı gruplarda daha fazla hata yapıyor mu?
- Yanlış tahminin etkisi nedir?
- Kullanıcı tahminin hata yapabileceğini biliyor mu?
- Model çıktısı gereğinden fazla kesin sunuluyor mu?

Yapay zeka geliştirmek yalnızca model skoru elde etmek değildir.


# 86. Bu Projede Öğrendiğimiz Python Bilgileri

Bu proje aslında önceki Python derslerinin birleşimidir:

### NumPy

- matris,
- indeksleme,
- reshape.

### Pandas

- DataFrame,
- analiz,
- gruplama,
- CSV.

### Matplotlib

- görüntü ve grafik.

### Fonksiyonlar

- tahmin fonksiyonları,
- rapor fonksiyonları.

### Dosya İşlemleri

- JSON,
- CSV,
- joblib.

### Makine Öğrenmesi

- model,
- Cross Validation,
- Grid Search,
- tahmin.


# 87. Proje Özeti

Bu derste:

- gerçek Digits veri kümesi,
- 8×8 görüntü,
- 64 piksel özelliği,
- çok sınıflı sınıflandırma,
- final test seti,
- baseline,
- Logistic Regression,
- KNN,
- Random Forest,
- SVM,
- Stratified K-Fold,
- Cross Validation,
- macro precision,
- macro recall,
- macro F1,
- GridSearchCV,
- final test,
- classification report,
- confusion matrix,
- hata analizi,
- sınıf bazlı başarı,
- decision function,
- PCA görselleştirme,
- model kaydetme,
- tahmin fonksiyonu,
- JSON proje raporu,
- CSV tahmin çıktısı,
- Flask/Tkinter/API entegrasyon planı

konularını tek bir yapay zeka projesinde birleştirdik.


# 88. Mini Uygulamalar

1. Digits veri kümesini yükleyin.
2. Veri kümesinin boyutunu bulun.
3. İlk 10 görüntünün gerçek rakamını yazdırın.
4. Beş farklı görüntüyü ayrı ayrı gösterin.
5. Sınıf dağılımını hesaplayın.
6. Sınıf dağılım grafiği oluşturun.
7. Eksik veri kontrolü yapın.
8. Final test setini %25 olarak ayırın.
9. DummyClassifier baseline oluşturun.
10. Logistic Regression Pipeline oluşturun.
11. KNN modeli oluşturun.
12. Random Forest modeli oluşturun.
13. SVM modeli oluşturun.
14. Bütün modeller için 5-fold Cross Validation yapın.
15. Accuracy karşılaştırma tablosu oluşturun.
16. Macro F1 karşılaştırma tablosu oluşturun.
17. En iyi model için GridSearchCV uygulayın.
18. Final test Accuracy hesaplayın.
19. Classification report oluşturun.
20. Confusion matrix oluşturun.
21. En çok karıştırılan rakam çiftlerini bulun.
22. Yanlış tahmin edilen beş görüntüyü gösterin.
23. Modeli `joblib` ile kaydedin.
24. Görüntüyü 64 özelliğe dönüştüren fonksiyon yazın.
25. Modeli yeniden yükleyip bir test görüntüsünü tahmin edin.


# 89. Bağımsız Yapay Zeka Proje Görevi

Bu dersin ardından öğrenciler kendi **uçtan uca makine öğrenmesi projesini** geliştirebilir.

Proje şu aşamaları içermelidir:

- problem tanımı,
- veri kümesi,
- özellikler,
- hedef,
- veri analizi,
- görselleştirme,
- final test seti,
- baseline,
- en az 3 model,
- Cross Validation,
- en az 2 değerlendirme metriği,
- model karşılaştırma,
- hiperparametre optimizasyonu,
- final test,
- hata analizi,
- yeni veri tahmini,
- model dosyasına kaydetme,
- kısa sonuç raporu.

Projeye ek olarak:

- Tkinter,
- Flask,
- SQLite,
- API

bileşenlerinden biri kullanılabilir.


# 90. Sonraki Aşama

Şimdiye kadar:

- sınıflandırma,
- regresyon,
- model değerlendirme,
- hiperparametre optimizasyonu,
- kümeleme,
- uçtan uca yapay zeka projesi

geliştirdik.

Bir sonraki dersimizde artık **Doğal Dil İşleme** alanına geçeceğiz.

İlk hedefimiz bilgisayarın metinleri nasıl sayısal verilere dönüştürdüğünü anlamak olacaktır.

Ardından:

```text
Metin
↓
Temizleme
↓
Sayısal Temsil
↓
Makine Öğrenmesi
↓
Metin Sınıflandırma
```

zincirini kuracağız.


# Dersin Ana Kazanımı

Bu dersin sonunda öğrencinin şu tam yapay zeka geliştirme sürecini kendi başına kurabilmesi hedeflenmektedir:

**Problem**

↓

**Gerçek Veri**

↓

**Veri Analizi**

↓

**Train / Final Test**

↓

**Baseline**

↓

**Birden Fazla Model**

↓

**Cross Validation**

↓

**Hiperparametre Optimizasyonu**

↓

**Final Test**

↓

**Hata Analizi**

↓

**Yeni Veri Tahmini**

↓

**Model Kaydı**

↓

**Uygulama Entegrasyonu**

Bu noktada öğrenciler yalnızca makine öğrenmesi komutlarını kullanan değil, uçtan uca bir yapay zeka projesinin temel aşamalarını planlayabilen ve uygulayabilen seviyeye gelmiş olacaktır.
